# 1. Imports

In [46]:
import os
import json
import random
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
     roc_curve
)

import matplotlib.pyplot as plt

from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# 2. Load Phase 1 Artifacts

In [25]:
# Environment setup
# Running in Colab with Google Drive (default):
BASE_DIR = Path("/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare")

# Running locally (PyCharm, Jupyter, VSCode):
# BASE_DIR = Path(".")

# Colab only: mount Google Drive
# Skip this cell if running locally
from google.colab import drive
drive.mount("/content/gdrive")

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [26]:
dataset_name = "heart_disease"
# dataset_name = "diabetes"
# dataset_name = "breast_cancer_wdbc"

ARTIFACT_DIR = BASE_DIR / "artifacts" / dataset_name
PHASE1_DIR = ARTIFACT_DIR / "phase1_model"
PHASE2_DIR = ARTIFACT_DIR / "phase2_candidate_groups"

PHASE2_DIR.mkdir(parents=True, exist_ok=True)

X_train_np = np.load(PHASE1_DIR / "X_train_np.npy")
X_test_np = np.load(PHASE1_DIR / "X_test_np.npy")

y_train_np = np.load(PHASE1_DIR / "y_train_np.npy")
y_test_np = np.load(PHASE1_DIR / "y_test_np.npy")

sample_ids_train = np.load(
    PHASE1_DIR / "sample_ids_train.npy",
    allow_pickle=True
)

sample_ids_test = np.load(
    PHASE1_DIR / "sample_ids_test.npy",
    allow_pickle=True
)

preprocessor = joblib.load(PHASE1_DIR / "preprocessor.joblib")

with open(PHASE1_DIR / "model_config.json", "r") as f:
    model_config = json.load(f)

X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

y_train_flat = y_train_np.ravel().astype(int)
y_test_flat = y_test_np.ravel().astype(int)

print("Loaded Phase 1 artifacts.")
print("Dataset:", dataset_name)
print("X_train:", X_train_np.shape)
print("X_test:", X_test_np.shape)
print("y_train:", y_train_flat.shape)
print("y_test:", y_test_flat.shape)
print("Model config:", model_config)

Loaded Phase 1 artifacts.
Dataset: heart_disease
X_train: (242, 28)
X_test: (61, 28)
y_train: (242,)
y_test: (61,)
Model config: {'dataset_name': 'heart_disease', 'original_file_name': 'processed.cleveland.data', 'input_size': 28, 'hidden1_size': 32, 'hidden2_size': 16, 'output_size': 1, 'loss_function': 'BCEWithLogitsLoss', 'optimizer': 'Adam', 'final_activation': 'linear_logit'}


# 3. Reload Trained PyTorch Model

In [27]:
class TabularMLP(nn.Module):
    def __init__(self, input_size):
        super(TabularMLP, self).__init__()

        self.hidden1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()

        self.hidden2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()

        self.output = nn.Linear(16, 1)

    def forward(self, x, return_activations=False):
        layer1 = self.relu1(self.hidden1(x))
        layer2 = self.relu2(self.hidden2(layer1))
        logits = self.output(layer2)

        if return_activations:
            return logits, {
                "layer1": layer1,
                "layer2": layer2
            }

        return logits


model = TabularMLP(input_size=model_config["input_size"])

model.load_state_dict(
    torch.load(
        PHASE1_DIR / "pytorch_model_state_dict.pt",
        map_location=torch.device("cpu")
    )
)

model.eval()

print("Reloaded trained PyTorch model.")

Reloaded trained PyTorch model.


# 4. Prepare Model Predictions

In [28]:
model.eval()

with torch.no_grad():
    train_logits = model(X_train_tensor)
    test_logits = model(X_test_tensor)

    train_probabilities = torch.sigmoid(train_logits).cpu().numpy().ravel()
    test_probabilities = torch.sigmoid(test_logits).cpu().numpy().ravel()

train_predictions = (train_probabilities >= 0.5).astype(int)
test_predictions = (test_probabilities >= 0.5).astype(int)

print("Prepared model probabilities and predictions.")
print("Train probabilities preview:", train_probabilities[:5])
print("Test probabilities preview:", test_probabilities[:5])

Prepared model probabilities and predictions.
Train probabilities preview: [0.9337743  0.07021048 0.27330524 0.17123945 0.80555034]
Test probabilities preview: [0.12335265 0.42564845 0.01767404 0.01438459 0.23706543]


# 5. Build Input and Activation Representation Spaces

In [29]:
input_train_representation = X_train_np
input_test_representation = X_test_np

model.eval()

with torch.no_grad():
    _, train_hidden_activations = model(
        X_train_tensor,
        return_activations=True
    )

    _, test_hidden_activations = model(
        X_test_tensor,
        return_activations=True
    )

train_layer1 = train_hidden_activations["layer1"].detach().cpu().numpy()
train_layer2 = train_hidden_activations["layer2"].detach().cpu().numpy()

test_layer1 = test_hidden_activations["layer1"].detach().cpu().numpy()
test_layer2 = test_hidden_activations["layer2"].detach().cpu().numpy()

layer1_scaler = StandardScaler()
layer2_scaler = StandardScaler()

train_layer1_scaled = layer1_scaler.fit_transform(train_layer1)
test_layer1_scaled = layer1_scaler.transform(test_layer1)

train_layer2_scaled = layer2_scaler.fit_transform(train_layer2)
test_layer2_scaled = layer2_scaler.transform(test_layer2)

activation_train_representation = np.concatenate(
    [train_layer1_scaled, train_layer2_scaled],
    axis=1
)

activation_test_representation = np.concatenate(
    [test_layer1_scaled, test_layer2_scaled],
    axis=1
)

representation_spaces = {
    "input": {
        "train": input_train_representation,
        "test": input_test_representation
    },
    "activation": {
        "train": activation_train_representation,
        "test": activation_test_representation
    }
}

print("Input train representation:", input_train_representation.shape)
print("Input test representation:", input_test_representation.shape)
print("Layer 1 train activations:", train_layer1.shape)
print("Layer 2 train activations:", train_layer2.shape)
print("Combined activation train representation:", activation_train_representation.shape)
print("Combined activation test representation:", activation_test_representation.shape)

Input train representation: (242, 28)
Input test representation: (61, 28)
Layer 1 train activations: (242, 32)
Layer 2 train activations: (242, 16)
Combined activation train representation: (242, 48)
Combined activation test representation: (61, 48)


# 6. Save Representation Artifacts

In [30]:
np.save(
    f"{PHASE2_DIR}/input_train_representation.npy",
    input_train_representation
)

np.save(
    f"{PHASE2_DIR}/input_test_representation.npy",
    input_test_representation
)

np.save(
    f"{PHASE2_DIR}/activation_train_representation.npy",
    activation_train_representation
)

np.save(
    f"{PHASE2_DIR}/activation_test_representation.npy",
    activation_test_representation
)

np.save(
    f"{PHASE2_DIR}/train_layer1_activations.npy",
    train_layer1
)

np.save(
    f"{PHASE2_DIR}/train_layer2_activations.npy",
    train_layer2
)

np.save(
    f"{PHASE2_DIR}/test_layer1_activations.npy",
    test_layer1
)

np.save(
    f"{PHASE2_DIR}/test_layer2_activations.npy",
    test_layer2
)

joblib.dump(layer1_scaler, f"{PHASE2_DIR}/layer1_scaler.joblib")
joblib.dump(layer2_scaler, f"{PHASE2_DIR}/layer2_scaler.joblib")

print("Saved Phase 2 representation artifacts to:")
print(PHASE2_DIR)

Saved Phase 2 representation artifacts to:
/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare/artifacts/heart_disease/phase2_candidate_groups


# 7. Generate Candidate KMeans Clusterings

In [31]:
all_kmeans_results = []
all_train_cluster_assignments = []
all_test_cluster_assignments = []

train_base_info = pd.DataFrame({
    "sample_id": sample_ids_train,
    "split": "train",
    "true_label": y_train_flat,
    "predicted_label": train_predictions,
    "predicted_probability": train_probabilities
})

test_base_info = pd.DataFrame({
    "sample_id": sample_ids_test,
    "split": "test",
    "true_label": y_test_flat,
    "predicted_label": test_predictions,
    "predicted_probability": test_probabilities
})

for space_name, reps in representation_spaces.items():
    train_representation = reps["train"]
    test_representation = reps["test"]

    max_k = min(10, train_representation.shape[0] - 1)

    for k in range(2, max_k + 1):
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        )

        train_cluster_labels = kmeans.fit_predict(train_representation)
        test_cluster_labels = kmeans.predict(test_representation)

        all_kmeans_results.append({
            "dataset": dataset_name,
            "representation_space": space_name,
            "k": k,
            "train_silhouette": silhouette_score(
                train_representation,
                train_cluster_labels
            ),
            "ari_true_label_train": adjusted_rand_score(
                y_train_flat,
                train_cluster_labels
            ),
            "nmi_true_label_train": normalized_mutual_info_score(
                y_train_flat,
                train_cluster_labels
            ),
            "ari_predicted_label_train": adjusted_rand_score(
                train_predictions,
                train_cluster_labels
            ),
            "nmi_predicted_label_train": normalized_mutual_info_score(
                train_predictions,
                train_cluster_labels
            )
        })

        train_assignments = train_base_info.copy()
        train_assignments["dataset"] = dataset_name
        train_assignments["representation_space"] = space_name
        train_assignments["k"] = k
        train_assignments["cluster"] = train_cluster_labels

        test_assignments = test_base_info.copy()
        test_assignments["dataset"] = dataset_name
        test_assignments["representation_space"] = space_name
        test_assignments["k"] = k
        test_assignments["cluster"] = test_cluster_labels

        all_train_cluster_assignments.append(train_assignments)
        all_test_cluster_assignments.append(test_assignments)

kmeans_summary = pd.DataFrame(all_kmeans_results)

train_cluster_assignments_long = pd.concat(
    all_train_cluster_assignments,
    ignore_index=True
)

test_cluster_assignments_long = pd.concat(
    all_test_cluster_assignments,
    ignore_index=True
)

cluster_assignments_long = pd.concat(
    [train_cluster_assignments_long, test_cluster_assignments_long],
    ignore_index=True
)

print("KMeans candidate summary:")
print(kmeans_summary.head())

print("\nCluster assignments preview:")
print(cluster_assignments_long.head())

KMeans candidate summary:
         dataset representation_space  k  train_silhouette  \
0  heart_disease                input  2          0.163747   
1  heart_disease                input  3          0.142777   
2  heart_disease                input  4          0.118881   
3  heart_disease                input  5          0.101069   
4  heart_disease                input  6          0.105970   

   ari_true_label_train  nmi_true_label_train  ari_predicted_label_train  \
0              0.188505              0.141421                   0.251052   
1              0.198173              0.158991                   0.239264   
2              0.168980              0.170744                   0.204455   
3              0.129371              0.144274                   0.166134   
4              0.123076              0.148550                   0.152530   

   nmi_predicted_label_train  
0                   0.190362  
1                   0.199376  
2                   0.208475  
3                   

# 8. Summarize Candidate Clusterings

In [32]:
candidate_cluster_summary = (
    cluster_assignments_long
    .groupby(["dataset", "split", "representation_space", "k", "cluster"])
    .agg(
        cluster_size=("sample_id", "count"),
        positive_true_label_rate=("true_label", "mean"),
        positive_prediction_rate=("predicted_label", "mean"),
        average_predicted_probability=("predicted_probability", "mean")
    )
    .reset_index()
)

candidate_cluster_summary["cluster_fraction"] = (
    candidate_cluster_summary["cluster_size"]
    / candidate_cluster_summary.groupby(
        ["dataset", "split", "representation_space", "k"]
    )["cluster_size"].transform("sum")
)

print("Candidate cluster summary:")
print(candidate_cluster_summary.head())

Candidate cluster summary:
         dataset split representation_space  k  cluster  cluster_size  \
0  heart_disease  test           activation  2        0            31   
1  heart_disease  test           activation  2        1            30   
2  heart_disease  test           activation  3        0            21   
3  heart_disease  test           activation  3        1            23   
4  heart_disease  test           activation  3        2            17   

   positive_true_label_rate  positive_prediction_rate  \
0                  0.064516                  0.000000   
1                  0.866667                  1.000000   
2                  0.000000                  0.000000   
3                  0.913043                  1.000000   
4                  0.411765                  0.411765   

   average_predicted_probability  cluster_fraction  
0                       0.116906          0.508197  
1                       0.902081          0.491803  
2                       0.038475

# 9. Build PCA Embeddings for Visualization

In [33]:
all_embedding_dfs = []

for space_name, reps in representation_spaces.items():
    train_representation = reps["train"]
    test_representation = reps["test"]

    pca = PCA(n_components=2, random_state=42)

    train_pca = pca.fit_transform(train_representation)
    test_pca = pca.transform(test_representation)

    train_embedding_df = pd.DataFrame({
        "sample_id": sample_ids_train,
        "dataset": dataset_name,
        "split": "train",
        "representation_space": space_name,
        "dim1": train_pca[:, 0],
        "dim2": train_pca[:, 1],
        "true_label": y_train_flat,
        "predicted_label": train_predictions,
        "predicted_probability": train_probabilities
    })

    test_embedding_df = pd.DataFrame({
        "sample_id": sample_ids_test,
        "dataset": dataset_name,
        "split": "test",
        "representation_space": space_name,
        "dim1": test_pca[:, 0],
        "dim2": test_pca[:, 1],
        "true_label": y_test_flat,
        "predicted_label": test_predictions,
        "predicted_probability": test_probabilities
    })

    embedding_df = pd.concat(
        [train_embedding_df, test_embedding_df],
        ignore_index=True
    )

    space_assignments = cluster_assignments_long[
        cluster_assignments_long["representation_space"] == space_name
    ]

    for k in sorted(space_assignments["k"].unique()):
        k_assignments = space_assignments[
            space_assignments["k"] == k
        ][["sample_id", "split", "cluster"]]

        k_assignments = k_assignments.rename(
            columns={"cluster": f"kmeans_k_{k}"}
        )

        embedding_df = embedding_df.merge(
            k_assignments,
            on=["sample_id", "split"],
            how="left"
        )

    all_embedding_dfs.append(embedding_df)

pca_embeddings = pd.concat(all_embedding_dfs, ignore_index=True)

print("PCA embeddings preview:")
print(pca_embeddings.head())

PCA embeddings preview:
   sample_id        dataset  split representation_space      dim1      dim2  \
0        180  heart_disease  train                input  0.500082 -0.105352   
1        208  heart_disease  train                input  1.196850  0.522227   
2        167  heart_disease  train                input  0.596645  1.202746   
3        105  heart_disease  train                input  1.110188  0.265300   
4        297  heart_disease  train                input -0.974706 -0.299899   

   true_label  predicted_label  predicted_probability  kmeans_k_2  kmeans_k_3  \
0           1                1               0.933774           0           0   
1           0                0               0.070210           0           0   
2           0                0               0.273305           0           1   
3           0                0               0.171239           0           0   
4           1                1               0.805550           1           2   

   kmeans_k_4 

# 10. Build Phase 3 SpArX Candidate Groups

In [34]:
sparx_candidate_groups = cluster_assignments_long.merge(
    candidate_cluster_summary,
    on=[
        "dataset",
        "split",
        "representation_space",
        "k",
        "cluster"
    ],
    how="left"
)

print("SpArX candidate groups preview:")
print(sparx_candidate_groups.head())

SpArX candidate groups preview:
   sample_id  split  true_label  predicted_label  predicted_probability  \
0        180  train           1                1               0.933774   
1        208  train           0                0               0.070210   
2        167  train           0                0               0.273305   
3        105  train           0                0               0.171239   
4        297  train           1                1               0.805550   

         dataset representation_space  k  cluster  cluster_size  \
0  heart_disease                input  2        0           129   
1  heart_disease                input  2        0           129   
2  heart_disease                input  2        0           129   
3  heart_disease                input  2        0           129   
4  heart_disease                input  2        1           113   

   positive_true_label_rate  positive_prediction_rate  \
0                  0.255814                  0.201550   


# 11. Save Phase 2 Candidate Clustering Outputs

In [35]:
kmeans_summary.to_csv(
    PHASE2_DIR / "kmeans_summary_by_representation.csv",
    index=False
)

cluster_assignments_long.to_csv(
    PHASE2_DIR / "cluster_assignments_long.csv",
    index=False
)

train_cluster_assignments_long.to_csv(
    PHASE2_DIR / "train_cluster_assignments_long.csv",
    index=False
)

test_cluster_assignments_long.to_csv(
    PHASE2_DIR / "test_cluster_assignments_long.csv",
    index=False
)

candidate_cluster_summary.to_csv(
    PHASE2_DIR / "cluster_summary_by_representation.csv",
    index=False
)

pca_embeddings.to_csv(
    PHASE2_DIR / "pca_embeddings_by_representation.csv",
    index=False
)

sparx_candidate_groups.to_csv(
    PHASE2_DIR / "sparx_candidate_groups.csv",
    index=False
)

print("Saved Phase 2 outputs to:")
print(PHASE2_DIR)
print(os.listdir(PHASE2_DIR))

Saved Phase 2 outputs to:
/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare/artifacts/heart_disease/phase2_candidate_groups
['activation_layer2_train_representation.npy', 'activation_layer2_test_representation.npy', 'activation_pca_train_representation.npy', 'activation_pca_test_representation.npy', 'hidden_activation_pca.joblib', 'input_train_representation.npy', 'activation_train_representation.npy', 'input_test_representation.npy', 'test_layer2_activations.npy', 'train_layer2_activations.npy', 'test_layer1_activations.npy', 'layer2_scaler.joblib', 'layer1_scaler.joblib', 'train_layer1_activations.npy', 'activation_test_representation.npy', 'kmeans_summary_by_representation.csv', 'test_cluster_assignments_long.csv', 'cluster_assignments_long.csv', 'train_cluster_assignments_long.csv', 'cluster_summary_by_representation.csv', 'pca_embeddings_by_representation.csv', 'sparx_candidate_groups.csv']


# Phase 3: Group-wise SpArX Explanations


# 12. Load Phase 2 Candidate Groups for SpArX

In [36]:
sparx_candidate_groups = pd.read_csv(
    PHASE2_DIR / "sparx_candidate_groups.csv"
)

print("Loaded SpArX candidate groups:")
print(sparx_candidate_groups.head())

print("\nAvailable representation spaces:")
print(sparx_candidate_groups["representation_space"].unique())

print("\nAvailable k values:")
print(sorted(sparx_candidate_groups["k"].unique()))

Loaded SpArX candidate groups:
   sample_id  split  true_label  predicted_label  predicted_probability  \
0        180  train           1                1               0.933774   
1        208  train           0                0               0.070210   
2        167  train           0                0               0.273305   
3        105  train           0                0               0.171239   
4        297  train           1                1               0.805550   

         dataset representation_space  k  cluster  cluster_size  \
0  heart_disease                input  2        0           129   
1  heart_disease                input  2        0           129   
2  heart_disease                input  2        0           129   
3  heart_disease                input  2        0           129   
4  heart_disease                input  2        1           113   

   positive_true_label_rate  positive_prediction_rate  \
0                  0.255814                  0.201550   
1

# 13. SpArX Installation and Compatibility Setup

In [37]:
import sys
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/SpArX")

subprocess.run(["rm", "-rf", str(REPO_DIR)], check=True)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/SpArX-Group-10/SpArX.git",
        str(REPO_DIR)
    ],
    check=True
)

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "scikit-learn",
        "bokeh",
        "networkx"
    ],
    check=True
)


ffnn_patch = r'''
from __future__ import annotations

import numpy as np
import tensorflow as tf


class FFNN:
    """Feed-forward neural network wrapper used by SpArX, patched for Colab/Keras 3."""

    def __init__(
        self,
        shape: tuple,
        weights: list[np.ndarray],
        bias: list[np.ndarray],
        activation_functions: list[str],
    ):
        self.shape = tuple(int(x) for x in shape)
        self.activation_functions = list(activation_functions)

        self.model = self._to_keras_model(
            self.shape,
            weights,
            bias,
            self.activation_functions,
        )

        self.output_model = self._create_output_model()
        self.functors = None
        self.data = None
        self.forward_pass_data = None

    def _to_keras_model(
        self,
        shape,
        weights,
        biases,
        activations,
    ) -> tf.keras.Model:
        shape = tuple(int(x) for x in shape)

        model = tf.keras.Sequential()
        model.add(tf.keras.Input(shape=(int(shape[0]),)))

        for i in range(1, len(shape)):
            model.add(
                tf.keras.layers.Dense(
                    units=int(shape[i]),
                    activation=activations[i - 1],
                )
            )

        model.compile(
            optimizer="adam",
            loss="binary_crossentropy",
            metrics=["accuracy"],
        )

        dense_layers = [
            layer for layer in model.layers
            if len(layer.get_weights()) == 2
        ]

        if len(dense_layers) != len(weights):
            raise ValueError(
                f"Layer mismatch: created {len(dense_layers)} Dense layers, "
                f"but received {len(weights)} weight matrices."
            )

        for layer, weight, bias in zip(dense_layers, weights, biases):
            layer.set_weights(
                [
                    np.asarray(weight, dtype=np.float32),
                    np.asarray(bias, dtype=np.float32),
                ]
            )

        return model

    def _create_output_model(self) -> tf.keras.Model:
        dense_layers = [
            layer for layer in self.model.layers
            if len(layer.get_weights()) == 2
        ]

        if len(dense_layers) == 0:
            return tf.keras.Model(
                inputs=self.model.inputs,
                outputs=self.model.inputs,
            )

        return tf.keras.Model(
            inputs=self.model.inputs,
            outputs=[layer.output for layer in dense_layers],
        )

    def forward_pass(self, inputs: np.ndarray) -> list[np.ndarray]:
        self.data = np.asarray(inputs, dtype=np.float32)

        outputs = self.output_model.predict(self.data, verbose=0)

        if not isinstance(outputs, list):
            outputs = [outputs]

        self.forward_pass_data = [np.asarray(output) for output in outputs]

        return self.forward_pass_data

    def add_layer(
        self,
        neuron_count: int,
        weights: np.ndarray,
        bias: np.ndarray,
        activation_function: str,
    ) -> None:
        self.activation_functions.append(activation_function)

        self.model.add(
            tf.keras.layers.Dense(
                units=int(neuron_count),
                activation=activation_function,
            )
        )

        dense_layers = [
            layer for layer in self.model.layers
            if len(layer.get_weights()) == 2
        ]

        dense_layers[-1].set_weights(
            [
                np.asarray(weights, dtype=np.float32),
                np.asarray(bias, dtype=np.float32),
            ]
        )

        self.output_model = self._create_output_model()

        if self.data is not None:
            self.forward_pass(self.data)

    def get_shape(self) -> tuple[int]:
        dense_layers = [
            layer for layer in self.model.layers
            if len(layer.get_weights()) == 2
        ]

        if len(dense_layers) == 0:
            return self.shape

        first_weight = dense_layers[0].get_weights()[0]
        input_dim = int(first_weight.shape[0])

        layer_units = [
            int(layer.get_weights()[0].shape[1])
            for layer in dense_layers
        ]

        return tuple([input_dim] + layer_units)
'''

(REPO_DIR / "sparx" / "ffnn.py").write_text(ffnn_patch)

merging_patch = r'''
from __future__ import annotations

from abc import abstractmethod

import numpy as np

from .ffnn import FFNN


class Merger:
    """Abstract class for merging algorithms."""

    @staticmethod
    @abstractmethod
    def merge(mlp: FFNN, cluster_labels: list[np.ndarray]) -> FFNN:
        raise NotImplementedError


class GlobalMerger(Merger):
    """Merges hidden neurons globally to construct a sparse SpArX model."""

    @staticmethod
    def merge(mlp: FFNN, cluster_labels: list[np.ndarray]) -> FFNN:
        dense_layers = [
            layer for layer in mlp.model.layers
            if len(layer.get_weights()) == 2
        ]

        if len(dense_layers) < 2:
            raise ValueError(
                "GlobalMerger expects at least one hidden layer "
                "and one output layer."
            )

        merged_weights = []
        merged_biases = []

        partial_weights = [dense_layers[0].get_weights()[0]]
        new_shape = []

        num_hidden_layers = len(dense_layers) - 1

        if len(cluster_labels) != num_hidden_layers:
            raise ValueError(
                f"Expected {num_hidden_layers} cluster-label arrays, "
                f"but received {len(cluster_labels)}."
            )

        for index in range(num_hidden_layers):
            labels = np.asarray(cluster_labels[index], dtype=int)
            number_of_clusters = int(np.max(labels) + 1)

            new_shape.append(number_of_clusters)

            merged_weights.append(
                GlobalMerger._merge_weights(
                    index=index,
                    num_clusters=number_of_clusters,
                    cluster_labels=cluster_labels,
                    partial_weights=partial_weights[index],
                )
            )

            merged_biases.append(
                GlobalMerger._merge_biases(
                    index=index,
                    mlp=mlp,
                    num_clusters=number_of_clusters,
                    cluster_labels=cluster_labels,
                )
            )

            partial_weights.append(
                GlobalMerger._update_partial_weights(
                    index=index,
                    mlp=mlp,
                    num_clusters=number_of_clusters,
                    cluster_labels=cluster_labels,
                )
            )

        merged_weights.append(partial_weights[-1])
        merged_biases.append(dense_layers[num_hidden_layers].get_weights()[1])

        old_shape = mlp.get_shape()

        new_shape = (
            [int(old_shape[0])]
            + [int(value) for value in new_shape]
            + [int(old_shape[-1])]
        )

        return FFNN(
            shape=tuple(new_shape),
            weights=merged_weights,
            bias=merged_biases,
            activation_functions=mlp.activation_functions[: len(new_shape) - 1],
        )

    @staticmethod
    def _merge_weights(
        index: int,
        num_clusters: int,
        cluster_labels: list[np.ndarray],
        partial_weights: np.ndarray,
    ) -> np.ndarray:
        labels = np.asarray(cluster_labels[index], dtype=int)

        new_layer_weights = []

        for label in range(num_clusters):
            new_layer_weights.append(
                np.mean(partial_weights.T[labels == label], axis=0)
            )

        return np.asarray(new_layer_weights, dtype=np.float32).T

    @staticmethod
    def _merge_biases(
        index: int,
        mlp: FFNN,
        num_clusters: int,
        cluster_labels: list[np.ndarray],
    ) -> np.ndarray:
        dense_layers = [
            layer for layer in mlp.model.layers
            if len(layer.get_weights()) == 2
        ]

        labels = np.asarray(cluster_labels[index], dtype=int)
        original_biases = dense_layers[index].get_weights()[1]

        new_layer_biases = []

        for label in range(num_clusters):
            new_layer_biases.append(
                np.mean(original_biases[labels == label])
            )

        return np.asarray(new_layer_biases, dtype=np.float32)

    @staticmethod
    def _update_partial_weights(
        index: int,
        mlp: FFNN,
        num_clusters: int,
        cluster_labels: list[np.ndarray],
    ) -> np.ndarray:
        dense_layers = [
            layer for layer in mlp.model.layers
            if len(layer.get_weights()) == 2
        ]

        labels = np.asarray(cluster_labels[index], dtype=int)
        next_weight_matrix = dense_layers[index + 1].get_weights()[0]

        new_partial_weights = []

        for label in range(num_clusters):
            new_partial_weights.append(
                np.sum(
                    next_weight_matrix[labels == label],
                    axis=0
                )
            )

        return np.asarray(new_partial_weights, dtype=np.float32)


class LocalMerger(Merger):
    """
    Kept only for compatibility with the SpArX package.
    It is not used in this notebook because the project evaluates GlobalMerger.
    """

    @staticmethod
    def merge(mlp: FFNN, cluster_labels: list[np.ndarray]) -> FFNN:
        raise NotImplementedError(
            "LocalMerger is not used in this notebook. "
            "Use GlobalMerger for the current evaluation."
        )
'''

(REPO_DIR / "sparx" / "merging.py").write_text(merging_patch)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

for module_name in list(sys.modules):
    if module_name == "sparx" or module_name.startswith("sparx."):
        del sys.modules[module_name]

from sparx import FFNN, KMeansClusterer
from sparx.merging import GlobalMerger

print("SpArX installation and compatibility setup completed successfully.")

SpArX installation and compatibility setup completed successfully.


# 14. Convert PyTorch Model to SpArX FFNN

In [38]:
def pytorch_model_to_sparx_ffnn(model):
    """
    Convert the trained PyTorch TabularMLP model into the FFNN format
    required by SpArX.

    Important:
    The output layer remains linear because the PyTorch model was trained
    with BCEWithLogitsLoss and therefore outputs logits.
    """
    model.eval()

    linear_layers = [
        model.hidden1,
        model.hidden2,
        model.output
    ]

    weights = [
        layer.weight.detach().cpu().numpy().T.astype(np.float32)
        for layer in linear_layers
    ]

    biases = [
        layer.bias.detach().cpu().numpy().astype(np.float32)
        for layer in linear_layers
    ]

    shape = tuple(
        [int(weights[0].shape[0])] +
        [int(weight.shape[1]) for weight in weights]
    )

    activation_functions = ["relu", "relu", "linear"]

    sparx_ffnn = FFNN(
        shape=shape,
        weights=weights,
        bias=biases,
        activation_functions=activation_functions
    )

    return sparx_ffnn


original_ffnn = pytorch_model_to_sparx_ffnn(model)

print("Converted PyTorch model to SpArX FFNN.")
print("SpArX FFNN shape:", original_ffnn.get_shape())

Converted PyTorch model to SpArX FFNN.
SpArX FFNN shape: (28, 32, 16, 1)


# 15. Validate PyTorch-to-SpArX Conversion

In [39]:
model.eval()

with torch.no_grad():
    pytorch_test_logits = model(X_test_tensor).detach().cpu().numpy().ravel()

sprax_test_logits = original_ffnn.forward_pass(X_test_np)[-1].ravel()

conversion_check_df = pd.DataFrame({
    "pytorch_logit": pytorch_test_logits,
    "sprax_logit": sprax_test_logits,
    "absolute_difference": np.abs(
        pytorch_test_logits - sprax_test_logits
    )
})

print("Conversion check preview:")
print(conversion_check_df.head())

print("\nMean absolute difference:")
print(conversion_check_df["absolute_difference"].mean())

print("\nMaximum absolute difference:")
print(conversion_check_df["absolute_difference"].max())

Conversion check preview:
   pytorch_logit  sprax_logit  absolute_difference
0      -1.961058    -1.961058         1.192093e-07
1      -0.299628    -0.299628         0.000000e+00
2      -4.017827    -4.017827         0.000000e+00
3      -4.227109    -4.227109         0.000000e+00
4      -1.168836    -1.168836         0.000000e+00

Mean absolute difference:
1.0259816e-07

Maximum absolute difference:
4.76837158203125e-07


# 16. Define SpArX Faithfulness and Complexity Metrics

In [40]:
def sigmoid_np(x):
    """
    Numerically stable sigmoid for NumPy arrays.
    Used for probability-space faithfulness evaluation.
    """
    x = np.asarray(x, dtype=np.float64)
    return 1.0 / (1.0 + np.exp(-x))


def input_output_unfaithfulness(
    original_ffnn,
    explanation_ffnn,
    X,
    output_space="probability",
    reduction="mean"
):
    """
    Measures how much the SpArX explanation model output differs
    from the original neural network output.

    output_space:
    - "probability": compare sigmoid(logits), recommended for this project
    - "logit": compare raw logits, useful as a diagnostic only
    """
    original_output = np.asarray(original_ffnn.forward_pass(X)[-1])
    explanation_output = np.asarray(explanation_ffnn.forward_pass(X)[-1])

    if original_output.shape != explanation_output.shape:
        raise ValueError(
            f"Output shape mismatch: {original_output.shape} "
            f"vs {explanation_output.shape}"
        )

    if output_space == "probability":
        original_output = sigmoid_np(original_output)
        explanation_output = sigmoid_np(explanation_output)

    elif output_space != "logit":
        raise ValueError("output_space must be 'probability' or 'logit'")

    per_sample = np.sum(
        (original_output - explanation_output) ** 2,
        axis=1
    )

    if reduction == "mean":
        return float(np.mean(per_sample))

    if reduction == "sum":
        return float(np.sum(per_sample))

    raise ValueError("reduction must be 'mean' or 'sum'")


def structural_unfaithfulness(
    original_ffnn,
    explanation_ffnn,
    X,
    neuron_cluster_labels,
    reduction="mean"
):
    """
    Measures structural unfaithfulness by comparing each original hidden-neuron
    activation with the activation of the merged SpArX neuron assigned to it.

    Lower values indicate better structural faithfulness.
    """
    original_activations = [
        np.asarray(activation)
        for activation in original_ffnn.forward_pass(X)
    ]

    explanation_activations = [
        np.asarray(activation)
        for activation in explanation_ffnn.forward_pass(X)
    ]

    per_sample_total = np.zeros(X.shape[0], dtype=np.float64)

    for layer_index, labels in enumerate(neuron_cluster_labels):
        labels = np.asarray(labels, dtype=int)

        original_layer = original_activations[layer_index]
        explanation_layer = explanation_activations[layer_index]

        if original_layer.shape[1] != labels.shape[0]:
            raise ValueError(
                f"Layer {layer_index}: labels length {labels.shape[0]} "
                f"does not match original neuron count {original_layer.shape[1]}."
            )

        if labels.max() >= explanation_layer.shape[1]:
            raise ValueError(
                f"Layer {layer_index}: label {labels.max()} is outside "
                f"merged layer with size {explanation_layer.shape[1]}."
            )

        merged_activation_for_each_original_neuron = explanation_layer[:, labels]

        per_sample_total += np.sum(
            (original_layer - merged_activation_for_each_original_neuron) ** 2,
            axis=1
        )

    if reduction == "mean":
        return float(np.mean(per_sample_total))

    if reduction == "sum":
        return float(np.sum(per_sample_total))

    raise ValueError("reduction must be 'mean' or 'sum'")


def cognitive_complexity(neuron_cluster_labels):
    """
    Simple complexity proxy based on the number of retained neuron groups.
    """
    complexity = 1

    for labels in neuron_cluster_labels:
        labels = np.asarray(labels)
        complexity *= int(np.max(labels) + 1)

    return complexity

# 17. Build Global SpArX Helper Function

In [41]:
def build_global_sparx_for_subset(
    original_ffnn,
    X_reference_subset,
    shrink_to_percentage,
    seed=42
):
    """
    Build a Global SpArX explanation model using a selected subset
    of patients as the reference data.
    """
    if X_reference_subset.shape[0] == 0:
        raise ValueError("Cannot build SpArX on an empty patient group.")

    original_ffnn.forward_pass(X_reference_subset)

    neuron_cluster_labels = KMeansClusterer.cluster(
        original_ffnn,
        shrink_to_percentage=shrink_to_percentage,
        seed=seed
    )

    global_ffnn = GlobalMerger.merge(
        original_ffnn,
        neuron_cluster_labels
    )

    return global_ffnn, neuron_cluster_labels

# 18. Build Global SpArX Baseline

In [42]:
SPARX_SHRINK_TO_PERCENTAGE = 0.5

baseline_global_ffnn, baseline_neuron_cluster_labels = build_global_sparx_for_subset(
    original_ffnn=original_ffnn,
    X_reference_subset=X_train_np,
    shrink_to_percentage=SPARX_SHRINK_TO_PERCENTAGE,
    seed=42
)

baseline_complexity = cognitive_complexity(
    baseline_neuron_cluster_labels
)

baseline_result = {
    "method": "Global SpArX baseline",
    "representation_space": "global",
    "patient_k": 1,
    "cluster": "all",
    "number_of_group_models": 1,
    "skipped_groups": 0,
    "sparx_shrink_percentage": SPARX_SHRINK_TO_PERCENTAGE,

    "train_group_size_min": X_train_np.shape[0],
    "train_group_size_max": X_train_np.shape[0],
    "test_group_size_min": X_test_np.shape[0],
    "test_group_size_max": X_test_np.shape[0],

    "complexity_sum": baseline_complexity,
    "complexity_mean": baseline_complexity,

    "train_io_unfaithfulness_probability_sum": input_output_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_train_np,
        output_space="probability",
        reduction="sum"
    ),
    "test_io_unfaithfulness_probability_sum": input_output_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_test_np,
        output_space="probability",
        reduction="sum"
    ),

    "train_io_unfaithfulness_logit_sum": input_output_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_train_np,
        output_space="logit",
        reduction="sum"
    ),
    "test_io_unfaithfulness_logit_sum": input_output_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_test_np,
        output_space="logit",
        reduction="sum"
    ),

    "train_structural_unfaithfulness_sum": structural_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_train_np,
        baseline_neuron_cluster_labels,
        reduction="sum"
    ),
    "test_structural_unfaithfulness_sum": structural_unfaithfulness(
        original_ffnn,
        baseline_global_ffnn,
        X_test_np,
        baseline_neuron_cluster_labels,
        reduction="sum"
    ),

    "evaluated_train_samples": X_train_np.shape[0],
    "evaluated_test_samples": X_test_np.shape[0],

    "model_shapes": [baseline_global_ffnn.get_shape()]
}

baseline_result["train_io_unfaithfulness_probability_mean"] = (
    baseline_result["train_io_unfaithfulness_probability_sum"]
    / baseline_result["evaluated_train_samples"]
)

baseline_result["test_io_unfaithfulness_probability_mean"] = (
    baseline_result["test_io_unfaithfulness_probability_sum"]
    / baseline_result["evaluated_test_samples"]
)

baseline_result["train_io_unfaithfulness_logit_mean"] = (
    baseline_result["train_io_unfaithfulness_logit_sum"]
    / baseline_result["evaluated_train_samples"]
)

baseline_result["test_io_unfaithfulness_logit_mean"] = (
    baseline_result["test_io_unfaithfulness_logit_sum"]
    / baseline_result["evaluated_test_samples"]
)

baseline_result["train_structural_unfaithfulness_mean"] = (
    baseline_result["train_structural_unfaithfulness_sum"]
    / baseline_result["evaluated_train_samples"]
)

baseline_result["test_structural_unfaithfulness_mean"] = (
    baseline_result["test_structural_unfaithfulness_sum"]
    / baseline_result["evaluated_test_samples"]
)

print("Global SpArX baseline result:")
for key, value in baseline_result.items():
    print(f"{key}: {value}")

Global SpArX baseline result:
method: Global SpArX baseline
representation_space: global
patient_k: 1
cluster: all
number_of_group_models: 1
skipped_groups: 0
sparx_shrink_percentage: 0.5
train_group_size_min: 242
train_group_size_max: 242
test_group_size_min: 61
test_group_size_max: 61
complexity_sum: 128
complexity_mean: 128
train_io_unfaithfulness_probability_sum: 0.33655909560896874
test_io_unfaithfulness_probability_sum: 0.07174156865417974
train_io_unfaithfulness_logit_sum: 12.462291717529297
test_io_unfaithfulness_logit_sum: 2.7759246826171875
train_structural_unfaithfulness_sum: 333.97388117201626
test_structural_unfaithfulness_sum: 84.35220348462462
evaluated_train_samples: 242
evaluated_test_samples: 61
model_shapes: [(28, 16, 8, 1)]
train_io_unfaithfulness_probability_mean: 0.0013907400644998708
test_io_unfaithfulness_probability_mean: 0.0011760912894127828
train_io_unfaithfulness_logit_mean: 0.051497073212930976
test_io_unfaithfulness_logit_mean: 0.04550696201011783
train_s

# 19. Define Group-wise SpArX Evaluation Function

In [43]:
def sample_id_mask(sample_ids_array, selected_sample_ids):
    """
    Robust sample-id matching for values loaded from CSV and NumPy.
    """
    sample_ids_as_string = pd.Series(sample_ids_array).astype(str)
    selected_ids_as_string = pd.Series(selected_sample_ids).astype(str)

    return sample_ids_as_string.isin(selected_ids_as_string).to_numpy()


def evaluate_groupwise_sparx_from_candidate_groups(
    original_ffnn,
    X_train,
    X_test,
    sample_ids_train,
    sample_ids_test,
    candidate_groups,
    representation_space,
    patient_k,
    shrink_to_percentage,
    seed=42,
    minimum_train_group_size=1
):
    """
    Evaluate group-wise SpArX using the clusters already created in Phase 2.
    """
    selected_groups = candidate_groups[
        (candidate_groups["representation_space"] == representation_space) &
        (candidate_groups["k"] == patient_k)
    ]

    train_groups = selected_groups[
        selected_groups["split"] == "train"
    ]

    test_groups = selected_groups[
        selected_groups["split"] == "test"
    ]

    train_io_probability_sum = 0.0
    test_io_probability_sum = 0.0

    train_io_logit_sum = 0.0
    test_io_logit_sum = 0.0

    train_structural_sum = 0.0
    test_structural_sum = 0.0

    group_complexities = []
    train_group_sizes = []
    test_group_sizes = []
    group_shapes = []
    skipped_groups = 0

    for cluster_id in sorted(train_groups["cluster"].unique()):
        train_cluster_rows = train_groups[
            train_groups["cluster"] == cluster_id
        ]

        test_cluster_rows = test_groups[
            test_groups["cluster"] == cluster_id
        ]

        train_mask = sample_id_mask(
            sample_ids_train,
            train_cluster_rows["sample_id"].values
        )

        test_mask = sample_id_mask(
            sample_ids_test,
            test_cluster_rows["sample_id"].values
        )

        X_train_group = X_train[train_mask]
        X_test_group = X_test[test_mask]

        train_group_size = int(X_train_group.shape[0])
        test_group_size = int(X_test_group.shape[0])

        if train_group_size < minimum_train_group_size:
            skipped_groups += 1
            continue

        group_ffnn, group_neuron_cluster_labels = build_global_sparx_for_subset(
            original_ffnn=original_ffnn,
            X_reference_subset=X_train_group,
            shrink_to_percentage=shrink_to_percentage,
            seed=seed
        )

        train_io_probability_sum += input_output_unfaithfulness(
            original_ffnn,
            group_ffnn,
            X_train_group,
            output_space="probability",
            reduction="sum"
        )

        train_io_logit_sum += input_output_unfaithfulness(
            original_ffnn,
            group_ffnn,
            X_train_group,
            output_space="logit",
            reduction="sum"
        )

        train_structural_sum += structural_unfaithfulness(
            original_ffnn,
            group_ffnn,
            X_train_group,
            group_neuron_cluster_labels,
            reduction="sum"
        )

        if test_group_size > 0:
            test_io_probability_sum += input_output_unfaithfulness(
                original_ffnn,
                group_ffnn,
                X_test_group,
                output_space="probability",
                reduction="sum"
            )

            test_io_logit_sum += input_output_unfaithfulness(
                original_ffnn,
                group_ffnn,
                X_test_group,
                output_space="logit",
                reduction="sum"
            )

            test_structural_sum += structural_unfaithfulness(
                original_ffnn,
                group_ffnn,
                X_test_group,
                group_neuron_cluster_labels,
                reduction="sum"
            )

        group_complexities.append(
            cognitive_complexity(group_neuron_cluster_labels)
        )

        train_group_sizes.append(train_group_size)
        test_group_sizes.append(test_group_size)
        group_shapes.append(group_ffnn.get_shape())

    evaluated_train_samples = int(np.sum(train_group_sizes))
    evaluated_test_samples = int(np.sum(test_group_sizes))

    return {
        "method": "Group-wise SpArX",
        "representation_space": representation_space,
        "patient_k": patient_k,
        "cluster": "all_groups",
        "number_of_group_models": len(group_shapes),
        "skipped_groups": skipped_groups,
        "sparx_shrink_percentage": shrink_to_percentage,

        "train_group_size_min": min(train_group_sizes) if train_group_sizes else np.nan,
        "train_group_size_max": max(train_group_sizes) if train_group_sizes else np.nan,
        "test_group_size_min": min(test_group_sizes) if test_group_sizes else np.nan,
        "test_group_size_max": max(test_group_sizes) if test_group_sizes else np.nan,

        "complexity_sum": int(np.sum(group_complexities)) if group_complexities else np.nan,
        "complexity_mean": float(np.mean(group_complexities)) if group_complexities else np.nan,

        "train_io_unfaithfulness_probability_sum": float(train_io_probability_sum),
        "test_io_unfaithfulness_probability_sum": float(test_io_probability_sum),

        "train_io_unfaithfulness_logit_sum": float(train_io_logit_sum),
        "test_io_unfaithfulness_logit_sum": float(test_io_logit_sum),

        "train_structural_unfaithfulness_sum": float(train_structural_sum),
        "test_structural_unfaithfulness_sum": float(test_structural_sum),

        "train_io_unfaithfulness_probability_mean": (
            train_io_probability_sum / evaluated_train_samples
            if evaluated_train_samples > 0 else np.nan
        ),
        "test_io_unfaithfulness_probability_mean": (
            test_io_probability_sum / evaluated_test_samples
            if evaluated_test_samples > 0 else np.nan
        ),

        "train_io_unfaithfulness_logit_mean": (
            train_io_logit_sum / evaluated_train_samples
            if evaluated_train_samples > 0 else np.nan
        ),
        "test_io_unfaithfulness_logit_mean": (
            test_io_logit_sum / evaluated_test_samples
            if evaluated_test_samples > 0 else np.nan
        ),

        "train_structural_unfaithfulness_mean": (
            train_structural_sum / evaluated_train_samples
            if evaluated_train_samples > 0 else np.nan
        ),
        "test_structural_unfaithfulness_mean": (
            test_structural_sum / evaluated_test_samples
            if evaluated_test_samples > 0 else np.nan
        ),

        "evaluated_train_samples": evaluated_train_samples,
        "evaluated_test_samples": evaluated_test_samples,
        "model_shapes": group_shapes
    }

# 20. Compare Global and Group-wise SpArX Across Candidate K Values

In [44]:
groupwise_results = []

representation_space_values = ["input", "activation"]
k_values = sorted(sparx_candidate_groups["k"].unique())

for representation_space in representation_space_values:
    for patient_k in k_values:
        print(
            f"Evaluating Group-wise SpArX: "
            f"{representation_space}, k={patient_k}"
        )

        result = evaluate_groupwise_sparx_from_candidate_groups(
            original_ffnn=original_ffnn,
            X_train=X_train_np,
            X_test=X_test_np,
            sample_ids_train=sample_ids_train,
            sample_ids_test=sample_ids_test,
            candidate_groups=sparx_candidate_groups,
            representation_space=representation_space,
            patient_k=patient_k,
            shrink_to_percentage=SPARX_SHRINK_TO_PERCENTAGE,
            seed=42,
            minimum_train_group_size=1
        )

        groupwise_results.append(result)

sprax_comparison_df = pd.DataFrame(
    [baseline_result] + groupwise_results
)

baseline_test_io_probability_sum = baseline_result[
    "test_io_unfaithfulness_probability_sum"
]

baseline_test_io_logit_sum = baseline_result[
    "test_io_unfaithfulness_logit_sum"
]

baseline_test_structural_sum = baseline_result[
    "test_structural_unfaithfulness_sum"
]

baseline_test_io_probability_mean = baseline_result[
    "test_io_unfaithfulness_probability_mean"
]

baseline_test_io_logit_mean = baseline_result[
    "test_io_unfaithfulness_logit_mean"
]

baseline_test_structural_mean = baseline_result[
    "test_structural_unfaithfulness_mean"
]

# Sum-based ratios
sprax_comparison_df["test_io_probability_sum_ratio_vs_global"] = (
    sprax_comparison_df["test_io_unfaithfulness_probability_sum"]
    / baseline_test_io_probability_sum
)

sprax_comparison_df["test_io_logit_sum_ratio_vs_global"] = (
    sprax_comparison_df["test_io_unfaithfulness_logit_sum"]
    / baseline_test_io_logit_sum
)

sprax_comparison_df["test_structural_sum_ratio_vs_global"] = (
    sprax_comparison_df["test_structural_unfaithfulness_sum"]
    / baseline_test_structural_sum
)

# Mean-based ratios are kept as diagnostics.
sprax_comparison_df["test_io_probability_mean_ratio_vs_global"] = (
    sprax_comparison_df["test_io_unfaithfulness_probability_mean"]
    / baseline_test_io_probability_mean
)

sprax_comparison_df["test_io_logit_mean_ratio_vs_global"] = (
    sprax_comparison_df["test_io_unfaithfulness_logit_mean"]
    / baseline_test_io_logit_mean
)

sprax_comparison_df["test_structural_mean_ratio_vs_global"] = (
    sprax_comparison_df["test_structural_unfaithfulness_mean"]
    / baseline_test_structural_mean
)

comparison_display_columns = [
    "method",
    "representation_space",
    "patient_k",
    "number_of_group_models",
    "skipped_groups",
    "complexity_sum",
    "complexity_mean",

    "test_io_unfaithfulness_probability_sum",
    "test_io_probability_sum_ratio_vs_global",
    "test_io_unfaithfulness_probability_mean",
    "test_io_probability_mean_ratio_vs_global",

    "test_io_unfaithfulness_logit_sum",
    "test_io_logit_sum_ratio_vs_global",
    "test_io_unfaithfulness_logit_mean",
    "test_io_logit_mean_ratio_vs_global",

    "test_structural_unfaithfulness_sum",
    "test_structural_sum_ratio_vs_global",
    "test_structural_unfaithfulness_mean",
    "test_structural_mean_ratio_vs_global",

    "train_group_size_min",
    "train_group_size_max",
    "test_group_size_min",
    "test_group_size_max",
    "evaluated_train_samples",
    "evaluated_test_samples"
]

comparison_display_df = (
    sprax_comparison_df[comparison_display_columns]
    .sort_values(
    by=[
        "test_io_probability_sum_ratio_vs_global",
        "test_structural_sum_ratio_vs_global"
    ]
)
    )

display(comparison_display_df)

Evaluating Group-wise SpArX: input, k=2
Evaluating Group-wise SpArX: input, k=3
Evaluating Group-wise SpArX: input, k=4
Evaluating Group-wise SpArX: input, k=5
Evaluating Group-wise SpArX: input, k=6
Evaluating Group-wise SpArX: input, k=7
Evaluating Group-wise SpArX: input, k=8
Evaluating Group-wise SpArX: input, k=9
Evaluating Group-wise SpArX: input, k=10
Evaluating Group-wise SpArX: activation, k=2
Evaluating Group-wise SpArX: activation, k=3
Evaluating Group-wise SpArX: activation, k=4
Evaluating Group-wise SpArX: activation, k=5
Evaluating Group-wise SpArX: activation, k=6
Evaluating Group-wise SpArX: activation, k=7
Evaluating Group-wise SpArX: activation, k=8
Evaluating Group-wise SpArX: activation, k=9
Evaluating Group-wise SpArX: activation, k=10


,method,representation_space,patient_k,number_of_group_models,skipped_groups,complexity_sum,complexity_mean,test_io_unfaithfulness_probability_sum,test_io_probability_sum_ratio_vs_global,test_io_unfaithfulness_probability_mean,...,test_structural_unfaithfulness_sum,test_structural_sum_ratio_vs_global,test_structural_unfaithfulness_mean,test_structural_mean_ratio_vs_global,train_group_size_min,train_group_size_max,test_group_size_min,test_group_size_max,evaluated_train_samples,evaluated_test_samples
2,Group-wise SpArX,input,3,3,0,384,128.0,0.053946,0.751954,0.000884,...,68.252974,0.809143,1.118901,0.809143,53,110,9,27,242,61
0,Global SpArX baseline,global,1,1,0,128,128.0,0.071742,1.000000,0.001176,...,84.352203,1.000000,1.382823,1.000000,242,242,61,61,242,61
1,Group-wise SpArX,input,2,2,0,256,128.0,0.094112,1.311825,0.001543,...,78.134195,0.926285,1.280888,0.926285,113,129,30,31,242,61
9,Group-wise SpArX,input,10,10,0,1280,128.0,0.098605,1.374452,0.001616,...,69.344214,0.822079,1.136790,0.822079,5,38,0,12,242,61
7,Group-wise SpArX,input,8,8,0,1024,128.0,0.150285,2.094808,0.002464,...,64.048470,0.759298,1.049975,0.759298,5,57,0,14,242,61
8,Group-wise SpArX,input,9,9,0,1152,128.0,0.229605,3.200441,0.003764,...,66.589375,0.789421,1.091629,0.789421,17,53,1,12,242,61
4,Group-wise SpArX,input,5,5,0,640,128.0,0.241159,3.361492,0.003953,...,76.252700,0.903980,1.250044,0.903980,28,67,7,17,242,61
11,Group-wise SpArX,activation,3,3,0,384,128.0,0.286313,3.990890,0.004694,...,76.503077,0.906948,1.254149,0.906948,73,92,17,23,242,61
6,Group-wise SpArX,input,7,7,0,896,128.0,0.305679,4.260841,0.005011,...,82.611660,0.979366,1.354290,0.979366,18,63,1,16,242,61
5,Group-wise SpArX,input,6,6,0,768,128.0,0.305879,4.263627,0.005014,...,78.125914,0.926187,1.280753,0.926187,21,71,3,17,242,61


# 21. Save Phase 3 SpArX Evaluation Results


In [45]:
PHASE3_DIR = ARTIFACT_DIR / "phase3_sparx_evaluation"
PHASE3_DIR.mkdir(parents=True, exist_ok=True)

sprax_comparison_df.to_csv(
    PHASE3_DIR / "global_vs_groupwise_sparx_comparison.csv",
    index=False
)

comparison_display_df.to_csv(
    PHASE3_DIR / "global_vs_groupwise_sparx_comparison_display.csv",
    index=False
)

print("Saved Phase 3 results to:")
print(PHASE3_DIR)
print(os.listdir(PHASE3_DIR))

Saved Phase 3 results to:
/content/gdrive/MyDrive/THESIS/thesis_xai_healthcare/artifacts/heart_disease/phase3_sparx_evaluation
['global_vs_groupwise_sprax_comparison.csv', 'global_vs_groupwise_sprax_comparison_display.csv', 'global_vs_groupwise_sparx_comparison_display.csv', 'global_vs_groupwise_sparx_comparison.csv']
